# Without Langchain - Do we had Harness?

In [2]:
import os
from getpass import  getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"]=getpass("Enter OPENAI_API_KEY:")

print("OpenAI API Key loaded!!")

OpenAI API Key loaded!!


# 1.1 Building a plain model

In [5]:
from openai import OpenAI
client = OpenAI()

SYSTEM_PROMPT = "You are a helpful assistant!!"

response=client.responses.create(model="gpt-4o-mini",instructions=SYSTEM_PROMPT,input="What's the weather in San Francisco?")

print(response.output_text)

#it would give the generic answer without the tool

I can't check real-time data, but you can find the current weather in San Francisco by checking a weather website or app for the latest updates. If you need general information about the climate or typical weather patterns in San Francisco, feel free to ask!


# 1.2 Now give the model a tool

In [11]:
from openai import OpenAI
import json

client = OpenAI()

SYSTEM_PROMPT = "You are a helpful assistant."

def get_weather(city: str) -> str:
    return f"It's always sunny in {city}!"

tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get weather for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string"
                }
            },
            "required": ["city"],
            "additionalProperties": False
        },
        "strict": True
    }
]

response = client.responses.create(
    model="gpt-4o-mini",
    instructions=SYSTEM_PROMPT,
    input="What's the weather in San Francisco?",
    tools=tools
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"city":"San Francisco"}', call_id='call_SFm9xmKbCg4JuZpaYGEv4oAO', name='get_weather', type='function_call', id='fc_012b1627a8db8ddf006ab0c4e6298887d18533c2aad64bc994', async_=None, caller=None, namespace=None, status='completed')]


# Important point is that the model didn't execute your Python fucntion.

Then?
It only requested that your application should execute it.

# This is the exact point the researchers introduced the word "Harness".


# 1.3 now let's build with Harness

In [18]:
from openai import OpenAI
import json

client=OpenAI()

SYSTEM_PROMPT="You are a helpful AI assistant."

#Step :1 - Create the tool
def get_weather(city: str) -> str:
    return f"It's always sunny in {city}!"

#Step 2: create the tool definition
tools=[{
    "type":"function",
    "name":"get_weather",
    "description":"Get weather for a given city",
    "parameters":{ "type":"object",
                   "properties":{"city":{"type":"string"}},
                   "required":["city"],
                   "additionalProperties":False},
    "strict":True}]

#Step 3: Create the Tool Map registry
tool_map={"get_weather":get_weather}

#Step 4:Get the user's request
input_items=[{"role":"user","content":"What's the weather in San Francisco?"}]

#Step 5:Create AGENT HARNESS
while True:
    #5.1. Ask the model what to do
    response=client.responses.create(model="gpt-4o-mini",
                                    instructions=SYSTEM_PROMPT,
                                    input=input_items,
                                    tools=tools)
    #5.2. Preserve the model's response
    input_items+=response.output

    #5.3. check whether the model requested the tool

    tool_calls=[item for item in response.output if item.type=="function_call"]

    #5.4 No tool call = Final Answer
    if not tool_calls:
        print(response.output_text)
        break

    #5.5. Execute requested tools
    for tool_call in tool_calls:
        print("MODEL->TOOL")
        print("Tool:",tool_call.name)
        print("Arguments:",tool_call.arguments)

        arguments = json.loads(tool_call.arguments)

        tool_function=tool_map[tool_call.name]

        result=tool_function(**arguments)

        print("TOOL -> MODEL")
        print("Result:",result)

        #5.56.We give the tool's result back to the model.

        input_items.append({"type":"function_call_output",
                     "call_id":tool_call.call_id,
                     "output":result})

MODEL->TOOL
Tool: get_weather
Arguments: {"city":"San Francisco"}
TOOL -> MODEL
Result: It's always sunny in San Francisco!
The weather in San Francisco is sunny! If you need more specific details, feel free to ask.


# What is Agent Loop?

Do it agin until the model no longer asks for a tool. That is the agent loop.

Model->Tool->Model->Tool...->until finished

# Till here is without Langchain Version of Building Harness.

# ================================================================

# 2.Langchain version - building Harness

In [ ]:
pip install -qU langchain "langchain[openai]"

In [22]:
from langchain.agents import create_agent

def get_weather(city:str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}"

agent=create_agent(
    model="gpt-4o-mini",
    tools=[get_weather],
    system_prompt="You are a helpful assistant"
)

result=agent.invoke({"messages":[{"role":"user","content":"What's the weather in San Francisco?"}]}
                   )

print(result["messages"][-1].content)

The weather in San Francisco is always sunny!


# The important lesson is that previously,we wrote the harness ourselves.Now LangChain is giving us that harness.